# 04 — Review Summarization with Llama3.2
**Project**: Automated Customer Reviews — IronHack

---

## Goal
Generate a **blog article per product category** that summarizes customer reviews into actionable product recommendations.

## Why Llama3.2 via Ollama?

| Reason | Detail |
|--------|--------|
| ✅ Explicitly mentioned in assignment | Project guidelines suggest *"llama mini (3B)"* as a model choice |
| ✅ 3B parameters | Within the 1B–13B range specified in the guidelines |
| ✅ Free & local | Runs on your own machine via Ollama — no API key or cloud costs |
| ✅ Instruction-following | Follows structured prompts to generate coherent blog articles |
| ✅ Best output quality | Outperformed FLAN-T5 and TinyLlama in our evaluation |

## Models Evaluated

We tested three models before selecting Llama3.2:

| Model | Output | Verdict |
|-------|--------|---------|
| `google/flan-t5-large` | Too short (< 30 chars) | ❌ Rejected |
| `TinyLlama/TinyLlama-1.1B-Chat-v1.0` | Hallucinated non-Amazon products | ❌ Rejected |
| `llama3.2` via Ollama | Professional structured articles | ✅ Selected |

## Prompt Engineering Techniques Applied

| Technique | Purpose |
|-----------|--------|
| Role assignment | `"You are a tech journalist"` — gives model context |
| Anti-hallucination | `"Only use these exact products"` — prevents inventing products |
| Explicit structure | `"REQUIRED sections: 1. 2. 3..."` — forces consistent output |
| Third person | `"Do NOT address the reviewer"` — prevents wrong tone |
| Product count | `"This category has exactly N products"` — prevents extra products |
| Worst product context | Rating + reviews included — helps model explain disappointment |

## Output per Category
- **Introduction** — short overview of the product category
- **Top 3 products** — pros, cons, top complaints, who should buy
- **Key differences** — explicit comparison product 1 vs 2 vs 3
- **Product to avoid** — worst rated product and why
- **Conclusion**

## Evaluation
| Metric | What it measures |
|--------|------------------|
| ROUGE-1 | Word overlap between generated text and source reviews |
| ROUGE-2 | Bigram overlap — measures fluency |
| ROUGE-L | Longest common sequence — measures coherence |

> **Note**: Lower ROUGE scores are not always bad — a good summary rewrites content in its own words rather than copying. This is expected for generative models like Llama3.2.

---

## 1. Imports

In [1]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ollama
from rouge_score import rouge_scorer

sns.set_theme(style='whitegrid', font_scale=1.15)

os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../data/plots',     exist_ok=True)

print('✅ Imports done')
print('If rouge_score is missing: pip install rouge-score')
print('If ollama is missing: pip install ollama')

✅ Imports done
If rouge_score is missing: pip install rouge-score
If ollama is missing: pip install ollama


## 2. Load Data

In [3]:
df = pd.read_csv('/Users/domiendarmont/Desktop/project_rev.aggre./data/processed/clustered_reviews.csv')

# Clean product names
df['product_name'] = (
    df['product_name']
    .str.split('\n').str[0]
    .str.replace(r',+$',  '', regex=True)
    .str.replace(r',,,+', '', regex=True)
    .str.strip()
)

print(f'Rows       : {len(df):,}')
print(f'Columns    : {df.columns.tolist()}')
print(f'Categories : {df["cluster_label"].unique().tolist()}')
df.head(3)

Rows       : 5,000
Columns    : ['product_name', 'brand', 'rating', 'review_title', 'review_text', 'ground_truth', 'cluster_id', 'cluster_label']
Categories : ['Fire Tablets', 'Batteries', 'Echo & Smart Speakers', 'Kindle E-readers', 'Fire TV & Accessories']


,product_name,brand,rating,review_title,review_text,ground_truth,cluster_id,cluster_label
0,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",Amazon,5,Excellent Tablet,Excellent tablet for everyday use. It is not a...,positive,4,Fire Tablets
1,AmazonBasics AAA Performance Alkaline Batterie...,Amazonbasics,5,Five Stars,Arrived on time. Long lasting batteries.,positive,1,Batteries
2,"All-New Fire HD 8 Tablet with Alexa, 8 HD Disp...",Amazon,5,Wonderful,Best Amazon tablet I've ever owned. While it c...,positive,4,Fire Tablets


## 3. Load Llama3.2 via Ollama

We use **Llama3.2 (3B parameters)** — a small but powerful open-source model from Meta, running locally via Ollama.

**Prerequisites** (one-time setup):
```bash
brew install ollama          # install Ollama
ollama pull llama3.2         # download the model (~2GB)
ollama serve                 # start the local server (keep this terminal open)
```

> ⚠️ Make sure `ollama serve` is running in a separate terminal before executing this cell.

In [5]:
def generator(prompt: str, max_new_tokens: int = 800) -> list:
    """Generate blog article using Ollama llama3.2 (3B parameters)."""
    response = ollama.chat(
        model    = 'llama3.2',
        messages = [{'role': 'user', 'content': prompt}],
        options  = {'temperature': 0.3, 'num_predict': max_new_tokens}
    )
    return [{'generated_text': response['message']['content'].strip()}]

# Quick test
test = generator('Write one sentence about Amazon Kindle.', max_new_tokens=50)
print(f'✅ Ollama llama3.2 ready')
print(f'Test output: {test[0]["generated_text"]}')

✅ Ollama llama3.2 ready
Test output: The Amazon Kindle is a popular e-reader device that allows users to access and read digital books, newspapers, magazines, and other content wirelessly, with features such as adjustable font sizes and built-in lighting.


## 4. Identify Top Products per Category

We rank products by a **weighted score**: `avg_rating × log(review_count)`

This rewards products that are both highly rated AND well-reviewed — a single 5-star review does not beat a product with 500 reviews at 4.5 stars.

In [6]:
def get_top_products(df_category: pd.DataFrame, n_top: int = 3) -> pd.DataFrame:
    """Rank products by weighted score: avg_rating x log(review_count)."""
    product_stats = (
        df_category.groupby('product_name')
        .agg(
            avg_rating   = ('rating', 'mean'),
            review_count = ('rating', 'count'),
        )
        .reset_index()
    )
    product_stats['score'] = (
        product_stats['avg_rating'] * np.log1p(product_stats['review_count'])
    )
    return product_stats.sort_values('score', ascending=False).head(n_top)

def get_worst_product(df_category: pd.DataFrame) -> str:
    """Find the worst rated product with enough reviews, excluding top 3."""
    product_stats = (
        df_category.groupby('product_name')
        .agg(
            avg_rating   = ('rating', 'mean'),
            review_count = ('rating', 'count'),
        )
        .reset_index()
    )
    filtered = product_stats[product_stats['review_count'] >= 20]
    if filtered.empty:
        filtered = product_stats[product_stats['review_count'] >= 5]

    top_products = get_top_products(df_category, n_top=3)['product_name'].tolist()
    filtered     = filtered[~filtered['product_name'].isin(top_products)]

    if filtered.empty or filtered['avg_rating'].isna().all():
        return 'Not enough data'
    return filtered.sort_values('avg_rating').iloc[0]['product_name']

# Preview top products per category
for category in df['cluster_label'].unique():
    df_cat = df[df['cluster_label'] == category]
    top    = get_top_products(df_cat)
    worst  = get_worst_product(df_cat)
    print(f'\n{category}:')
    for _, row in top.iterrows():
        print(f'  ⭐ {row["avg_rating"]:.1f} ({row["review_count"]} reviews) — {row["product_name"][:60]}')
    print(f'  ❌ Worst: {worst[:60]}')


Fire Tablets:
  ⭐ 4.5 (1176 reviews) — Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes Special Offer
  ⭐ 4.6 (321 reviews) — All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi, 16 GB - Inclu
  ⭐ 4.5 (178 reviews) — Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16 GB, Green Kid
  ❌ Worst: Brand New Amazon Kindle Fire 16gb 7 Ips Display Tablet Wifi 

Batteries:
  ⭐ 4.3 (735 reviews) — AmazonBasics AAA Performance Alkaline Batteries (36 Count)
  ⭐ 4.4 (148 reviews) — AmazonBasics AA Performance Alkaline Batteries (48 Count) - 
  ❌ Worst: Not enough data

Echo & Smart Speakers:
  ⭐ 4.6 (340 reviews) — Echo (White)
  ⭐ 4.7 (94 reviews) — Amazon Echo Show Alexa-enabled Bluetooth Speaker with 7" Scr
  ⭐ 4.7 (59 reviews) — Amazon - Echo Plus w/ Built-In Hub - Silver
  ❌ Worst: Amazon - Amazon Tap Portable Bluetooth and Wi-Fi Speaker - B

Kindle E-readers:
  ⭐ 4.8 (323 reviews) — Amazon Kindle Paperwhite - eBook reader - 4 GB - 6 monochrom
  ⭐ 4.8 (66 reviews) — Kindle Voyage E-reader, 6 High-Resoluti

## 5. Generate Blog Articles with Llama3.2

For each category we:
1. Select top 3 products + worst product
2. Sample 15 reviews per product as context
3. Apply prompt engineering to guide Llama3.2
4. Generate a structured blog article

**Key prompt engineering techniques:**
- Role assignment: `"You are a tech journalist"`
- Anti-hallucination: `"Only use these exact products"`
- Explicit structure: `"REQUIRED sections: 1. 2. 3..."`
- Third person: `"Do NOT address the reviewer"`

> ⏱️ Expected time: ~2-3 minutes per category on Apple Silicon.

In [ ]:
N_REVIEWS_PER_PRODUCT = 15
MAX_NEW_TOKENS        = 800

def get_product_reviews_text(df_category: pd.DataFrame, product_name: str) -> str:
    """Get sample reviews for a specific product as one long text."""
    subset  = df_category[df_category['product_name'] == product_name]
    reviews = (
        subset['review_text']
        .dropna()
        .sample(min(N_REVIEWS_PER_PRODUCT, len(subset)), random_state=42)
        .tolist()
    )
    return ' '.join([r[:200] for r in reviews])

def get_top_complaints_text(df_category: pd.DataFrame, product_name: str) -> str:
    """Get negative reviews (rating ≤ 3) for a product as one long text."""
    subset = df_category[
        (df_category['product_name'] == product_name) &
        (df_category['rating'] <= 3)
    ]
    if subset.empty:
        return 'No significant complaints found.'
    complaints = (
        subset['review_text']
        .dropna()
        .sample(min(5, len(subset)), random_state=42)
        .tolist()
    )
    return ' '.join([r[:200] for r in complaints])

def generate_blog_article(category: str, df_category: pd.DataFrame) -> str:
    """Generate a structured blog article using Ollama llama3.2."""
    top_products  = get_top_products(df_category)
    worst_product = get_worst_product(df_category)

    # Exact product names — prevents hallucination
    product_names = [row['product_name'][:60] for _, row in top_products.iterrows()]
    n_products    = len(product_names)

    product_info = ''
    for _, row in top_products.iterrows():
        reviews    = get_product_reviews_text(df_category, row['product_name'])
        complaints = get_top_complaints_text(df_category, row['product_name'])
        product_info += (
            f"Product: {row['product_name'][:60]} "
            f"(rating: {row['avg_rating']:.1f}/5, {row['review_count']} reviews). "
            f"Reviews: {reviews[:150]}. "
            f"Complaints: {complaints[:100]}. "
        )

    worst_avg_rating = (
        df_category[df_category['product_name'] == worst_product]['rating'].mean()
        if worst_product != 'Not enough data' else None
    )
    worst_reviews = (
        get_product_reviews_text(df_category, worst_product)
        if worst_product != 'Not enough data' else ''
    )
    worst_section = (
        f"Product to avoid: {worst_product[:60]} (rating: {worst_avg_rating:.1f}/5). "
        f"Based on these reviews: {worst_reviews[:150]}. Explain why customers were disappointed. "
        if worst_product != 'Not enough data'
        else "Product to avoid: All products in this category are well-rated. No product to avoid. "
    )

    # ── Prompt engineering ───────────────────────────────────
    prompt = (
        f"You are a tech journalist. Write a professional product recommendation blog article for the category: {category}. "
        f"Do NOT address the reviewer directly. Write FOR the reader. Use third person only. "
        f"This category has exactly {n_products} products. Only use these exact products — do not invent others: {', '.join(product_names)}. "
        f"REQUIRED sections: "
        f"1. Introduction. "
        f"2. Top {n_products} products with pros, cons, complaints, who should buy. "
        f"3. Key differences: compare products explicitly. When to choose which? "
        f"4. {worst_section} "
        f"5. Conclusion. "
        f"Product data: {product_info[:400]}."
    )
    prompt = prompt[:1024]

    result = generator(prompt, max_new_tokens=MAX_NEW_TOKENS)
    return result[0]['generated_text'].strip()

# Generate articles for all categories
articles = {}
for category in df['cluster_label'].unique():
    print(f'\n📝 Generating article for: {category}...')
    df_cat  = df[df['cluster_label'] == category]
    article = generate_blog_article(category, df_cat)
    articles[category] = article
    print(f'✅ Done ({len(article)} characters)')


📝 Generating article for: Fire Tablets...


## 6. Preview Articles

In [ ]:
for category, article in articles.items():
    print(f'\n{"="*60}')
    print(f'ARTICLE: {category}')
    print(f'{"="*60}\n')
    print(article)
    print()

## 7. Evaluate with ROUGE

We compare each generated article against the source reviews it was based on.

**Interpreting scores**:
- **ROUGE-1 > 0.2** → meaningful content overlap with source reviews ✅
- **Low ROUGE-2** → model rewrites in own words — expected for generative models ✅
- **ROUGE-L > 0.1** → coherent sentence structure ✅

> **Note**: Llama3.2 scores slightly lower than extractive models like BART because it generates creative text rather than copying from reviews. This is a deliberate trade-off for better readability.

In [ ]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_results = []
for category, article in articles.items():
    reference = ' '.join(
        df[df['cluster_label'] == category]['review_text']
        .dropna()
        .sample(min(50, len(df[df['cluster_label'] == category])), random_state=42)
        .tolist()
    )
    scores = scorer.score(reference, article)
    rouge_results.append({
        'category' : category,
        'rouge1'   : scores['rouge1'].fmeasure,
        'rouge2'   : scores['rouge2'].fmeasure,
        'rougeL'   : scores['rougeL'].fmeasure,
    })

rouge_df = pd.DataFrame(rouge_results)
print('ROUGE scores per category:')
print(rouge_df.to_string(index=False))
print(f'\nAverage ROUGE-1 : {rouge_df["rouge1"].mean():.3f}')
print(f'Average ROUGE-2 : {rouge_df["rouge2"].mean():.3f}')
print(f'Average ROUGE-L : {rouge_df["rougeL"].mean():.3f}')

In [ ]:
# ── Plot: ROUGE scores per category ─────────────────────────
x     = np.arange(len(rouge_df))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - width, rouge_df['rouge1'], width, label='ROUGE-1', color='#3498db', edgecolor='white')
ax.bar(x,          rouge_df['rouge2'], width, label='ROUGE-2', color='#2ecc71', edgecolor='white')
ax.bar(x + width,  rouge_df['rougeL'], width, label='ROUGE-L', color='#e74c3c', edgecolor='white')

ax.set_title(
    'ROUGE Scores per Product Category\n'
    '(measures overlap between generated article and source reviews)',
    fontsize=12, pad=12
)
ax.set_xticks(x)
ax.set_xticklabels(rouge_df['category'], rotation=15, ha='right', fontsize=9)
ax.set_ylabel('ROUGE Score', fontsize=11)
ax.set_ylim(0, 1)
ax.axhline(y=0.2, color='grey', linestyle='--', linewidth=1, label='Typical minimum (0.2)')
ax.legend(fontsize=9)
sns.despine()

plt.tight_layout()
plt.savefig('../data/plots/13_rouge_scores.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save Articles

In [ ]:
# Save all articles in one CSV file
articles_df = pd.DataFrame([
    {'category': category, 'article': article}
    for category, article in articles.items()
])

OUTPUT_PATH = '../data/processed/articles.csv'
articles_df.to_csv(OUTPUT_PATH, index=False)

print(f'✅ All articles saved to: {OUTPUT_PATH}')
print(articles_df[['category']].to_string(index=False))

# Save ROUGE scores
rouge_df.to_csv('../data/processed/rouge_scores.csv', index=False)
print('\n✅ ROUGE scores saved to ../data/processed/rouge_scores.csv')

## 9. Summary

| Step | Detail |
|------|--------|
| Model | `llama3.2` (3B parameters) via Ollama |
| Why chosen | Best output quality among FLAN-T5, TinyLlama, and Llama3.2 |
| Articles generated | One per product category |
| Product ranking | Weighted score: avg_rating × log(review_count) |
| Reviews per product | 15 sample reviews as context |
| Top complaints | Extracted from reviews with rating ≤ 3 |
| Prompt engineering | Role assignment · Anti-hallucination · Explicit structure · Third person |
| Evaluation | ROUGE-1 · ROUGE-2 · ROUGE-L |
| Output | `../data/processed/articles.csv` |

➡️ **Next**: `05_deployment.py` — Streamlit web app showing classifications, clusters, and blog articles with product images.